# Sleep EDA — Polar H10 overnight ECG

Exploratory analysis of one night of Polar H10 ECG, using this repository's own
sleep-staging code. Point `CSV_PATH` at your sleep log and run all cells.

### About "Sleep²"

**Sleep² (Topalidis et al. 2023, *Sensors* 23(5):2390) is not code in this
repository** — it is a proprietary network that cannot run here. What this
project takes from it is two things:

1. The canonical **4-class Wake / Light / Deep / REM** display vocabulary
   (`StageVocab.WAKE_LIGHT_DEEP_REM`, `app/sleep/stages.py`).
2. Its H10-validated accuracy — **80.3 %, κ≈0.69** — as the honest ceiling that
   every engine's accuracy note is measured against.

The engine that produces a hypnogram in that Sleep² vocabulary is this repo's
built-in **`heuristic`** engine: transparent rules over per-epoch heart rate,
RMSSD, LF/HF and movement, with a Viterbi pass for temporal structure
(`app/sleep/engines/heuristic.py`). This notebook runs it alongside the
**`sleepecg`** pre-trained GRU (3-class Wake/NREM/REM) and compares the two,
because where two engines disagree is exactly where a single number should not
be trusted.

### Setup

```
polarh10/Scripts/python -m pip install -r requirements-sleep.txt -r requirements-notebook.txt
```

### What this is not

> Sleep stages here are estimated from heartbeat patterns (and movement, when
> an accelerometer file is attached), not from brain activity. Even the best
> published heart-beat-based model validated on this strap reaches ~80% epoch
> agreement with laboratory polysomnography (Topalidis et al. 2023, Sensors
> 23(5):2390) — treat every number as an estimate with real error. This
> analysis cannot detect sleep apnea, periodic limb movements, or the
> difference between quiet wakefulness and sleep misperception, and none of
> its output is a diagnosis or a substitute for a sleep study.

---
## 0. Configuration

The only cell you need to edit. Leave `CSV_PATH` empty to list the sleep logs
already on disk.

In [14]:
pip install -r requirements-sleep.txt -r requirements-notebook.txt

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements-sleep.txt'


In [18]:
# --- edit these ------------------------------------------------------------
REPO_ROOT = r"c:\Users\Ferhat\Documents\GitHub\polarh10-ecg-logger"

CSV_PATH = r"c:\Users\Ferhat\Documents\GitHub\polarh10-ecg-logger\data\uploads\ferhat-culfaz\11__20-08-2026_23_56_slwep.csv"          # your Polar H10 ECG sleep log; "" lists what is on disk
ACC_PATH = None         # optional Polar accelerometer export (str path or None)

AGE_YEARS = 44        # improves the sleepecg classifier a little
SEX = "male"              # "male" | "female" | None — a classifier covariate only

PREFER_CACHE = True     # reuse the .npz written at processing time when present
FOCUS_ENGINE = "heuristic"   # engine used by the single-engine sections below
# ---------------------------------------------------------------------------

---
## 1. Environment and engines

Which engines can actually run here. An engine that is missing says so up
front rather than quietly vanishing from the results.

In [19]:
import datetime as dt
import sys
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# sleep_eda_core lives beside this notebook; it puts the repo root on the
# path so `import app...` works in the cells below.
sys.path.insert(0, str(Path(REPO_ROOT) / "notebooks"))
import sleep_eda_core as core  # noqa: E402

core.ensure_repo_on_path(REPO_ROOT)

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 4)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
pd.set_option("display.width", 140)

# Palette copied rather than imported: `app.report.figures` calls
# matplotlib.use("Agg") at import time, which would disable inline plotting.
# Values mirror the ECG-paper palette there (app/report/figures.py:29-35).
PAPER, INK, ACCENT, WARN, MUTED = "#FFFCFA", "#1B1E28", "#2E5E4E", "#B4472E", "#8A8F9E"

# One colour per stage label across every vocabulary the app speaks.
STAGE_COLORS = {
    "Wake": "#B4472E", "REM": "#7B4B94", "Light": "#4C7FA8", "Deep": "#22415C",
    "NREM": "#2E5E4E", "Sleep": "#2E5E4E",
    "N1": "#7FA8C4", "N2": "#4C7FA8", "N3": "#22415C",
    "Unscored": "#C9CCD4",
}

# Clinical y-axis order per vocabulary size: Wake on top, then REM, then
# progressively deeper NREM. Mirrors _CLINICAL_ORDER in
# app/report/sleep_figures.py:31 so notebook and report read the same way.
CLINICAL_ORDER = {2: [0, 1], 3: [0, 2, 1], 4: [0, 3, 1, 2], 5: [0, 4, 1, 2, 3]}

for status in core.engine_statuses():
    mark = "available" if status.available else f"unavailable — {status.unavailable_reason}"
    print(f"{status.key:<16} {status.label}\n{'':<16} {mark}\n")

heuristic        Built-in rules (cardio-actigraphy)
                 available

sleepecg         SleepECG GRU (wrn-gru-mesa-weighted)
                 available

external-5class  External 5-class deep net (adammj/ecg-sleep-staging)
                 unavailable — not configured — clone the AGPL tool separately and set ECGLOG_SLEEP_EXTERNAL_DIR and ECGLOG_SLEEP_EXTERNAL_PYTHON (see docs/SLEEP.md).



In [ ]:
# --- TEMPORARY diagnostic: why does TensorFlow's DLL load fail only inside
# this kernel process, when it loads fine via the same python.exe from a
# terminal? Delete this cell once the sleepecg engine works again.
import ctypes
import os
import sys

print("sys.executable   ", sys.executable)
print("cwd              ", os.getcwd())
print("PATH entries containing 'python' or 'tensorflow':")
for p in os.environ.get("PATH", "").split(os.pathsep):
    if "python" in p.lower() or "tensorflow" in p.lower():
        print("   ", p)

dll_dir = r"C:\Users\Ferhat\AppData\Local\Python\pythoncore-3.12-64\python312.dll"
try:
    ctypes.WinDLL(dll_dir)
    print(f"\nExplicit full-path load of python312.dll: OK ({dll_dir})")
except OSError as e:
    print(f"\nExplicit full-path load of python312.dll FAILED: {e!r}")

tf_common = (
    Path(REPO_ROOT) / "polarh10" / "Lib" / "site-packages" / "tensorflow"
    / "python" / "_pywrap_tensorflow_common.dll"
)
try:
    ctypes.WinDLL(str(tf_common))
    print(f"Explicit full-path load of _pywrap_tensorflow_common.dll: OK")
except OSError as e:
    print(f"Explicit full-path load of _pywrap_tensorflow_common.dll FAILED: {e!r}")

---
## 2. Load the night

**Cache-first.** When the app has already processed this CSV it wrote a `.npz`
beside it holding the corrected RR series, the R-peak train and the per-window
quality summaries. Those are the only inputs staging needs, so the cache branch
skips reading the ~100 MB waveform and loads in milliseconds. Delete the
`.npz`, or set `PREFER_CACHE = False`, to force the full `load_polar_csv` →
`run_pipeline` path (several minutes for a full night).

One caveat worth stating out loud: the `QualityResult` rebuilt from the cache
carries the per-window summaries but not the per-sample SQI and wander traces,
which are not stored. It is complete for staging and for nothing else.

In [20]:
if not CSV_PATH:
    candidates = core.find_candidate_csvs(REPO_ROOT)
    if not candidates:
        raise FileNotFoundError(
            "No CSVs found under data/uploads. Set CSV_PATH to your Polar H10 sleep log."
        )
    print("Set CSV_PATH to one of these (largest first — an overnight file is big):\n")
    for p in candidates[:20]:
        cached = "cached" if core.cache_path_for_csv(p).exists() else "      "
        print(f"  {p.stat().st_size / 1e6:8.1f} MB  {cached}  {p}")
    raise SystemExit("CSV_PATH is empty — pick a file above and re-run this cell.")

data = core.load_session(CSV_PATH, prefer_cache=PREFER_CACHE)
print(f"source     {data.source}")
for note in data.notes:
    print(f"           - {note}")

source     cache
           - Epoch auto-detect: unix interpretation lands at 2026-08-20 22:56 (plausible); no date found in filename to cross-check.
           - Sampling rate 130.05 Hz estimated from the first 2000 samples (the loader derives it over the whole file).
           - Loaded from the processing cache 11__20-08-2026_23_56_slwep.npz (version 3); the raw ECG waveform was not read.


---
## 3. Run the staging engines

Both engines score the same canonical 30 s epoch grid starting at the first ECG
sample, and both mark epochs they cannot score `Unscored` rather than guessing.
If an accelerometer file was given, sustained movement forces those epochs to
Wake in *every* engine's hypnogram (`app/sleep/actigraphy.py`).

Every number below comes from `app.sleep` — `summarize`, `stage_stats`,
`pairwise_agreement`. Nothing is recomputed here.

In [21]:
analysis, features = core.stage_night(
    data, acc_path=ACC_PATH, age_years=AGE_YEARS, sex=SEX
)
if not analysis.hypnograms:
    for note in analysis.notes:
        print(note)
    raise SystemExit("No engine produced a hypnogram — see the notes above.")

epochs = core.epoch_table(data, analysis, features)
focus = analysis.hypnogram_for(FOCUS_ENGINE) or analysis.hypnograms[0]

print(f"engines run       {', '.join(h.engine for h in analysis.hypnograms)}")
print(f"primary engine    {analysis.primary_engine}  (precedence, not quality)")
print(f"focus engine      {focus.engine} — {focus.engine_label}")
print(f"movement source   {features.movement_source}")
print(f"epochs            {features.n_epochs} x {focus.epoch_len_s:.0f} s")


[TensorFlow DLL Diagnostic] Analyzing: c:\Users\Ferhat\Documents\GitHub\polarh10-ecg-logger\polarh10\Lib\site-packages\tensorflow\python\_pywrap_tensorflow_internal.pyd
[Error] Failed to load python312.dll: UNKNOWN ERROR (None): Could not find module 'python312.dll' (or one of its dependencies). Try using the full path with constructor syntax.
[Error] Failed to load _pywrap_tensorflow_common.dll: UNKNOWN ERROR (None): Could not find module '_pywrap_tensorflow_common.dll' (or one of its dependencies). Try using the full path with constructor syntax.
engines run       heuristic
primary engine    heuristic  (precedence, not quality)
focus engine      heuristic — Built-in rules (cardio-actigraphy)
movement source   wander_proxy
epochs            811 x 30 s


---
## 4. Recording overview

Coverage first: staging is only as good as the beats underneath it, so how much
of the night carried usable RR intervals is the first thing to look at.

In [ ]:
from app.sleep.epochs import MIN_EPOCH_COVERAGE

start_local = data.start_time.astimezone()
end_local = start_local + dt.timedelta(seconds=data.duration_s)
excluded_s = data.quality.excluded_total_s

overview = pd.Series({
    "File": Path(data.csv_path).name,
    "Started (local)": f"{start_local:%Y-%m-%d %H:%M:%S}",
    "Ended (local)": f"{end_local:%Y-%m-%d %H:%M:%S}",
    "Duration": f"{data.duration_s / 3600:.2f} h",
    "Sampling rate": (
        "unknown" if data.sampling_rate_hz is None else f"{data.sampling_rate_hz:.2f} Hz"
    ),
    "R-peaks detected": f"{len(data.peak_times_s):,}",
    "RR intervals used": f"{len(data.rr):,}",
    "RR discontinuities": (f"{int(np.sum(data.rr.discontinuity)):,} "
                           f"({np.mean(data.rr.discontinuity) * 100:.2f} %)"),
    "Signal excluded": (f"{excluded_s / 60:.1f} min "
                        f"({excluded_s / data.duration_s * 100:.1f} %)"),
    "Mean epoch RR coverage": f"{np.nanmean(features.coverage) * 100:.1f} %",
    "Epochs below coverage floor": (
        f"{int(np.sum(features.coverage < MIN_EPOCH_COVERAGE))} of "
        f"{features.n_epochs} (<{MIN_EPOCH_COVERAGE:.0%})"
    ),
    "Unstageable epochs": f"{int(np.sum(~features.stageable()))}",
}, name="")
overview.to_frame()

---
## 5. Per-epoch features — what the stager actually sees

The heuristic engine decides from these five series and nothing else. Reading
them top to bottom is the fastest way to understand *why* a night was scored
the way it was — and to spot when a decision rests on a feature that was
missing or noisy.

Physiological expectations (`app/sleep/engines/heuristic.py`): **deep sleep** is
the night's vagal maximum — lowest HR, LF/HF minimum; **REM** is sympathetic
predominance — LF/HF maximum with almost no gross movement; **wake** is
movement, or HR well above the sustained nocturnal baseline.

In [ ]:
from app.sleep.engines.heuristic import (
    DEEP_HR_DELTA_MAX_BPM,
    DEEP_LFHF_MAX,
    REM_LFHF_MIN,
    WAKE_HR_DELTA_BPM,
)

t = epochs["clock_time"]
baseline_hr = float(np.nanmean(features.mean_hr_bpm - features.hr_vs_baseline))

fig, axes = plt.subplots(5, 1, figsize=(13, 12), sharex=True)

axes[0].plot(t, features.mean_hr_bpm, color=INK, lw=0.9)
axes[0].axhline(baseline_hr, color=ACCENT, ls="--", lw=1,
                label=f"night baseline {baseline_hr:.1f} bpm (10th pct)")
axes[0].axhline(baseline_hr + WAKE_HR_DELTA_BPM, color=WARN, ls=":", lw=1,
                label=f"wake evidence (+{WAKE_HR_DELTA_BPM:.0f} bpm)")
axes[0].axhline(baseline_hr + DEEP_HR_DELTA_MAX_BPM, color="#22415C", ls=":", lw=1,
                label=f"deep-sleep ceiling (+{DEEP_HR_DELTA_MAX_BPM:.0f} bpm)")
axes[0].set_ylabel("mean HR\n(bpm)")
axes[0].legend(fontsize=7, loc="upper right", ncol=3)

axes[1].plot(t, features.rmssd_ms, color=ACCENT, lw=0.9)
axes[1].set_ylabel("RMSSD\n(ms, 5-min window)")

axes[2].plot(t, features.lf_hf, color="#7B4B94", lw=0.9)
axes[2].set_yscale("log")
axes[2].axhline(DEEP_LFHF_MAX, color="#22415C", ls=":", lw=1,
                label=f"deep below {DEEP_LFHF_MAX:g}")
axes[2].axhline(REM_LFHF_MIN, color="#7B4B94", ls=":", lw=1,
                label=f"REM above {REM_LFHF_MIN:g}")
axes[2].set_ylabel("LF/HF\n(5-min window)")
axes[2].legend(fontsize=7, loc="upper right", ncol=2)

axes[3].plot(t, features.movement, color=MUTED, lw=0.9)
if np.isfinite(features.movement_threshold):
    axes[3].axhline(features.movement_threshold, color=WARN, ls=":", lw=1,
                    label="movement threshold (median + 5·MAD)")
    axes[3].legend(fontsize=7, loc="upper right")
unit = "activity counts (mg·s)" if features.movement_source == "acc" else "wander RMS (mV)"
axes[3].set_ylabel(f"movement\n{unit}")

axes[4].fill_between(t, features.coverage, color=ACCENT, alpha=0.35)
axes[4].axhline(MIN_EPOCH_COVERAGE, color=WARN, ls=":", lw=1,
                label=f"stageable floor ({MIN_EPOCH_COVERAGE:.0%})")
axes[4].set_ylabel("RR coverage")
axes[4].set_ylim(0, 1.05)
axes[4].legend(fontsize=7, loc="lower right")

axes[4].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
axes[4].set_xlabel("clock time (local)")
fig.suptitle(f"Per-epoch features — {Path(data.csv_path).name}"
             f"  (movement source: {features.movement_source})", y=0.995)
fig.tight_layout()
plt.show()

if features.movement_source != "acc":
    print("Note: no accelerometer file — movement is ECG baseline wander, a coarse")
    print("proxy that misses still-body wakefulness. Attach an ACC export to improve it.")

---
## 6. Hypnograms

Clinical convention: Wake on top, then REM, then progressively deeper NREM.
Each engine's accuracy note is printed beneath the figure — the estimate and
its error travel together, always.

In [ ]:
import textwrap

from app.sleep.stages import UNSCORED, stage_labels

fig, axes = plt.subplots(len(analysis.hypnograms), 1,
                         figsize=(13, 2.6 * len(analysis.hypnograms)),
                         sharex=True, squeeze=False)
for ax, hyp in zip(axes[:, 0], analysis.hypnograms, strict=True):
    labels = stage_labels(hyp.vocab)
    order = CLINICAL_ORDER[len(labels)]
    y_of_code = {c: pos for pos, c in enumerate(order)}
    y = np.array([y_of_code.get(int(s), np.nan) for s in hyp.stages], dtype=float)

    ax.step(t, y, where="post", color=INK, lw=1.0)
    for c, label in enumerate(labels):
        ax.fill_between(t, y_of_code[c] - 0.42, y_of_code[c] + 0.42,
                        where=(hyp.stages == c), step="post",
                        color=STAGE_COLORS[label], alpha=0.75)
    for i in np.flatnonzero(hyp.stages == UNSCORED):
        ax.axvspan(t.iloc[i], t.iloc[min(i + 1, len(t) - 1)],
                   color=STAGE_COLORS["Unscored"], alpha=0.7)

    ax.set_yticks(range(len(order)))
    ax.set_yticklabels([labels[c] for c in order])
    ax.set_ylim(len(order) - 0.5, -0.5)
    n_over = analysis.override_epochs_n.get(hyp.engine, 0)
    extra = f"  ·  {n_over} epoch(s) forced to Wake by movement" if n_over else ""
    n_unscored = int(np.sum(hyp.stages == UNSCORED))
    unscored = f"  ·  {n_unscored} unscored (grey)" if n_unscored else ""
    ax.set_title(f"{hyp.engine_label}  ({hyp.vocab.value}){extra}{unscored}",
                 fontsize=9, loc="left")

axes[-1, 0].xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
axes[-1, 0].set_xlabel("clock time (local)")
fig.tight_layout()
plt.show()

for hyp in analysis.hypnograms:
    print(f"\n{hyp.engine_label}")
    print(textwrap.fill(hyp.accuracy_note, 94, initial_indent="  ", subsequent_indent="  "))
    for note in hyp.notes:
        print(textwrap.fill(f"- {note}", 94, initial_indent="  ", subsequent_indent="    "))

---
## 7. Sleep architecture

Definitions are the app's, each an epoch count × 0.5 min
(`app/sleep/summary.py`): time in bed is the whole scored grid — the recording
start stands in for lights-off; sleep onset is the first epoch of any sleep
stage (AASM v2.6); WASO is wake strictly between onset and the last sleep
epoch, so terminal wake is not counted as WASO.

Stage minutes are vocabulary-honest: a 3-class engine reports no deep-sleep
figure, and none is invented for it — those cells stay empty.

In [ ]:
ROWS = [
    ("Time in bed (min)", "tib_min"), ("Total sleep time (min)", "tst_min"),
    ("Sleep efficiency (%)", "sleep_efficiency_pct"),
    ("Sleep onset latency (min)", "sol_min"), ("WASO (min)", "waso_min"),
    ("Awakenings", "awakenings_n"), ("REM latency (min)", "rem_latency_min"),
    ("Unscored (min)", "unscored_min"), ("Wake (min)", "wake_min"),
    ("Light (min)", "light_min"), ("Deep (min)", "deep_min"), ("REM (min)", "rem_min"),
    ("NREM (min)", "nrem_min"), ("N1 (min)", "n1_min"), ("N2 (min)", "n2_min"),
]
arch = pd.DataFrame({eng: {label: getattr(s, attr) for label, attr in ROWS}
                     for eng, s in analysis.summaries.items()})
display(arch.dropna(how="all").round(1))

pct = pd.DataFrame({eng: s.stage_pct_of_tst
                    for eng, s in analysis.summaries.items()}).T.fillna(0.0)
ax = pct.plot(kind="barh", stacked=True, figsize=(11, 1.1 * len(pct) + 1.8),
              color=[STAGE_COLORS.get(c, MUTED) for c in pct.columns])
ax.set_xlabel("% of total sleep time")
ax.set_xlim(0, 100)
ax.set_title("Stage composition (% of TST) — vocabularies differ between engines",
             fontsize=10)
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

---
## 8. Physiology by stage

A sanity check with teeth. If the staging is tracking real autonomic state,
**Deep** should hold the lowest heart rate and the highest RMSSD, and **REM**
the highest LF/HF. If it does not, the hypnogram is labelling noise.

RMSSD here never differences across a stage boundary or a dropped interval —
that would mix autonomic states (`app/sleep/summary.py`).

In [ ]:
for eng, rows in analysis.stage_stats.items():
    print(f"\n{eng}")
    display(pd.DataFrame(rows).set_index("stage").round(1))

col = f"stage_{focus.engine}"
present = [lab for lab in stage_labels(focus.vocab) + ["Unscored"]
           if (epochs[col] == lab).any()]

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (field_, title) in zip(axes, [("mean_hr_bpm", "mean HR (bpm)"),
                                      ("rmssd_ms", "RMSSD (ms)"),
                                      ("lf_hf", "LF/HF")], strict=True):
    groups = [epochs.loc[epochs[col] == lab, field_].dropna().to_numpy() for lab in present]
    bp = ax.boxplot(groups, tick_labels=present, patch_artist=True, showfliers=False)
    for patch, lab in zip(bp["boxes"], present, strict=True):
        patch.set_facecolor(STAGE_COLORS.get(lab, MUTED))
        patch.set_alpha(0.65)
    for median in bp["medians"]:
        median.set_color(INK)
    ax.set_title(title, fontsize=10)
    ax.tick_params(axis="x", rotation=30)
axes[2].set_yscale("log")
fig.suptitle(f"Per-epoch distributions by stage — {focus.engine_label}", fontsize=10)
fig.tight_layout()
plt.show()

---
## 9. Engine agreement

Where two engines agree, the night's numbers are worth reading as values.
Where they disagree, read them as a range. Comparison happens in the coarser of
the two vocabularies — detail is merged, never invented, so the 4-class
heuristic collapses to the GRU's Wake/NREM/REM rather than the reverse.

Rough Cohen's κ reading (Landis & Koch 1977): <0.20 slight, 0.21–0.40 fair,
0.41–0.60 moderate, 0.61–0.80 substantial.

In [ ]:
from app.sleep.stages import common_vocab

if not analysis.agreement:
    print("Only one engine produced a hypnogram — nothing to compare.")

for r in analysis.agreement:
    display(pd.DataFrame([{
        "engine A": r.engine_a, "engine B": r.engine_b, "vocabulary": r.vocab.value,
        "Cohen's kappa": r.kappa, "% agreement": r.percent_agree,
        "epochs compared": r.n_epochs,
    }]).round(3))
    if r.kappa is None:
        print("Too few mutually scored epochs for a meaningful kappa.")
        continue

    ha, hb = analysis.hypnogram_for(r.engine_a), analysis.hypnogram_for(r.engine_b)
    vocab = common_vocab(ha.vocab, hb.vocab)
    a, b = ha.collapsed(vocab).stages, hb.collapsed(vocab).stages
    both = (a != UNSCORED) & (b != UNSCORED)
    names = stage_labels(vocab)
    cm = np.zeros((len(names), len(names)))
    np.add.at(cm, (a[both].astype(int), b[both].astype(int)), 1)
    share = cm / max(cm.sum(), 1) * 100

    fig, ax = plt.subplots(figsize=(5.4, 4.6))
    im = ax.imshow(share, cmap="BuPu", vmin=0)
    for i in range(len(names)):
        for j in range(len(names)):
            ax.text(j, i, f"{int(cm[i, j])}\n{share[i, j]:.1f}%", ha="center",
                    va="center", fontsize=8,
                    color=INK if share[i, j] < 0.55 * share.max() else PAPER)
    ax.set_xticks(range(len(names)), names)
    ax.set_yticks(range(len(names)), names)
    ax.set_xlabel(r.engine_b)
    ax.set_ylabel(r.engine_a)
    ax.set_title(f"Epoch confusion in {vocab.value}\n"
                 f"kappa={r.kappa:.3f}, {r.percent_agree:.1f}% agree", fontsize=9)
    ax.grid(False)
    fig.colorbar(im, ax=ax, label="% of compared epochs")
    fig.tight_layout()
    plt.show()

    if r.kappa < 0.40:
        print(f"Low agreement (kappa={r.kappa:.2f}). The heavy off-diagonal cells above")
        print("show exactly where the two engines part company. Read this night's stage")
        print("minutes as a range, not a value.")

---
## 10. Structure over the night

Real sleep has structure: it cycles roughly every 90 minutes, deep sleep is
front-loaded, REM builds toward morning. Staging that produces plausible stage
*totals* with implausible *structure* — no bouts, no cycles, a flat
distribution across the night — is a sign the classifier is following a slow
drift in the signal rather than sleep itself.

The heuristic engine carries a REM circadian prior, so the last panel is
checking one of its own assumptions, not independently confirming it.

In [ ]:
codes = focus.stages
labels_f = stage_labels(focus.vocab)

# --- transitions between consecutive scored epochs -------------------------
scored = codes != UNSCORED
pair = scored[:-1] & scored[1:]
trans = np.zeros((len(labels_f), len(labels_f)), dtype=int)
np.add.at(trans, (codes[:-1][pair].astype(int), codes[1:][pair].astype(int)), 1)
print(f"Stage transitions (rows = from, columns = to) — {focus.engine_label}")
display(pd.DataFrame(trans, index=labels_f, columns=labels_f))
changes = int(trans.sum() - np.trace(trans))
print(f"{changes} stage changes across {int(pair.sum())} epoch boundaries "
      f"({changes / max(int(pair.sum()), 1) * 100:.1f} % — a stable night changes rarely)")

# --- bout lengths ----------------------------------------------------------
bouts, run_start, run_code = [], 0, codes[0]
for i in range(1, len(codes) + 1):
    if i == len(codes) or codes[i] != run_code:
        if run_code != UNSCORED:
            bouts.append({"stage": labels_f[int(run_code)],
                          "minutes": (i - run_start) * focus.epoch_len_s / 60.0})
        if i < len(codes):
            run_start, run_code = i, codes[i]
bouts_df = pd.DataFrame(bouts)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
present_b = [lab for lab in labels_f if (bouts_df["stage"] == lab).any()]
bp = axes[0].boxplot([bouts_df.loc[bouts_df["stage"] == lab, "minutes"].to_numpy()
                      for lab in present_b], tick_labels=present_b, patch_artist=True)
for patch, lab in zip(bp["boxes"], present_b, strict=True):
    patch.set_facecolor(STAGE_COLORS.get(lab, MUTED))
    patch.set_alpha(0.65)
for median in bp["medians"]:
    median.set_color(INK)
axes[0].set_yscale("log")
axes[0].set_ylabel("bout length (min, log scale)")
axes[0].set_title(f"Bout lengths per stage ({len(bouts_df)} bouts)", fontsize=10)

# --- composition by third of night ----------------------------------------
third = np.minimum((np.arange(len(codes)) * 3) // len(codes), 2)
comp = pd.DataFrame({
    name: pd.Series([int(np.sum(codes[(third == k) & scored] == c))
                     for c in range(len(labels_f))], index=labels_f)
          / max(int(np.sum((third == k) & scored)), 1) * 100
    for k, name in enumerate(["1st third", "2nd third", "3rd third"])
}).T
comp.plot(kind="bar", stacked=True, ax=axes[1],
          color=[STAGE_COLORS.get(c, MUTED) for c in comp.columns])
axes[1].set_ylabel("% of scored epochs")
axes[1].set_title("Composition by third of night", fontsize=10)
axes[1].legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc="upper left")
axes[1].tick_params(axis="x", rotation=0)
fig.tight_layout()
plt.show()

display(comp.round(1))
print("Expected in a normal night: deep sleep concentrated in the 1st third, REM")
print("building toward the 3rd. A flat table means the engine found no cycles.")

---
## 11. Plausibility check

Every engine here reports its accuracy honestly, but an accuracy note cannot
tell you whether *this particular night* came out sensible. This cell compares
the result against adult normative ranges (Ohayon et al. 2017, *Sleep Health*
3(1):6-19; AASM scoring conventions) and says so plainly when it does not fit.

A result outside these ranges is not automatically wrong — but it is the first
thing to explain before quoting any number from this notebook elsewhere.

In [ ]:
# (low, high) as % of total sleep time, healthy adult.
NORM_PCT = {"Light": (40, 65), "N1": (2, 10), "N2": (40, 60), "Deep": (5, 25),
            "N3": (5, 25), "REM": (15, 28), "NREM": (72, 85)}
NORM_OTHER = [("Sleep efficiency (%)", "sleep_efficiency_pct", 85, 100),
              ("Sleep onset latency (min)", "sol_min", 0, 30),
              ("WASO (min)", "waso_min", 0, 40),
              ("REM latency (min)", "rem_latency_min", 60, 120)]

checks = []
for eng, s in analysis.summaries.items():
    for label, value in s.stage_pct_of_tst.items():
        if label in NORM_PCT:
            checks.append((eng, f"{label} (% of TST)", float(value), *NORM_PCT[label]))
    for label, attr, low, high in NORM_OTHER:
        value = getattr(s, attr)
        if value is not None:
            checks.append((eng, label, float(value), low, high))

report = pd.DataFrame(checks, columns=["engine", "metric", "value", "low", "high"])
report["verdict"] = np.where(
    (report["value"] >= report["low"]) & (report["value"] <= report["high"]),
    "within range",
    np.where(report["value"] < report["low"], "BELOW range", "ABOVE range"))
display(report.round(1))

flagged = report[report["verdict"] != "within range"]
if flagged.empty:
    print("Every metric sits inside the normative range.")
else:
    print(f"{len(flagged)} metric(s) outside the normative range:\n")
    for _, r in flagged.iterrows():
        print(f"  [{r['engine']}] {r['metric']} = {r['value']:.1f} "
              f"({r['verdict'].lower()}, expected {r['low']:g}-{r['high']:g})")
    print("\nCommon explanations, in rough order of likelihood:")
    print("  - Cardio-only staging genuinely confuses these classes. REM and Light are")
    print("    the pair most often swapped; Wake and N1 are the least reliable classes.")
    print("  - No accelerometer file, so still-body wakefulness was scored as sleep.")
    print("  - Recording started well before lights-off, inflating time in bed and SOL.")
    print("  - Low RR coverage over the affected stretch (see section 4).")
    print("\nWhere two engines disagree strongly (section 9), prefer neither — the honest")
    print("read is a range spanning both.")

---
## 12. Notes, limitations, and export

Everything the loader, the features and the engines wanted to tell you,
collected in one place, followed by a tidy per-epoch table written to disk for
whatever you want to do next.

In [ ]:
def section(title, items):
    if not items:
        return
    print(f"\n{title}")
    for item in items:
        print(textwrap.fill(f"- {item}", 94, initial_indent="  ", subsequent_indent="    "))

section("Loading", data.notes)
section("Feature extraction", features.notes)
section("Analysis", analysis.notes)
for hyp in analysis.hypnograms:
    section(f"{hyp.engine_label} — accuracy", [hyp.accuracy_note])
    section(f"{hyp.engine_label} — notes", hyp.notes)
section("Engines not run", [f"{s.label}: {s.unavailable_reason}"
                            for s in analysis.engines if not s.available])

print("\n" + "=" * 94)
print(textwrap.fill(
    "Sleep stages here are estimated from heartbeat patterns (and movement, when an "
    "accelerometer file is attached), not from brain activity. Even the best published "
    "heart-beat-based model validated on this strap reaches ~80% epoch agreement with "
    "laboratory polysomnography (Topalidis et al. 2023, Sensors 23(5):2390) — treat every "
    "number as an estimate with real error. This analysis cannot detect sleep apnea, "
    "periodic limb movements, or the difference between quiet wakefulness and sleep "
    "misperception, and none of its output is a diagnosis or a substitute for a sleep "
    "study.", 94))
print("=" * 94)

out_dir = Path(REPO_ROOT) / "notebooks" / "output"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / f"{Path(data.csv_path).stem}_epochs.csv"
epochs.to_csv(out_path, index=False)
print(f"\nPer-epoch table written to {out_path}")
display(epochs.head())